 Automated Problem-Solving Pipeline Using LangChain and LLM Models

 This script automates the extraction, solution, execution, and evaluation
 of algorithmic problems (e.g., LeetCode problems) by interfacing with
 large language models (LLMs) from both Google and OpenAI.

 The pipeline performs the following steps:
   1. Installs required packages using pip magic commands.
   2. Imports necessary libraries for file handling, time measurement, and 
      interaction with LLM models.
   3. Configures parameters for two different model providers:
      - Google Generative AI models.
      - OpenAI models via LangChain.
   4. Sets up directories for storing output files and results.
   5. Reads a CSV file containing problem descriptions.
   6. For a subset of problems, constructs prompts, interacts with the chosen 
      LLM to obtain code solutions and test examples, and then:
         a. Extracts and cleans the Python code.
         b. Saves the code to a file.
         c. Executes the saved code using a subprocess.
         d. Parses the output to count occurrences of "True" and "False".
         e. Logs the results for further analysis.
   7. Writes the accumulated results to a CSV file.

Step 0: Install and Upgrade Required Packages

In [ ]:
# The following magic commands ensure that the necessary packages are installed.
# Note: These commands may require a kernel restart to take effect.
%pip install --upgrade --quiet langchain pandas numpy matplotlib seaborn jupyter
%pip install --upgrade --quiet langchain-google-genai openai langchain-openai
%pip install --upgrade --quiet langchain_community

^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Step 1: Import Libraries

In [ ]:
import csv
import os
import time
import subprocess
import openai
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain.chat_models import ChatOpenAI

Step 2: Configure Model Settings for Google Generative AI

In [ ]:
# List of available Google models and selection of a specific model.
llm_models = ["gemini-1.0-pro", "gemini-1.5-pro", "gemini-1.5-flash"]
llm_model = llm_models[0]
google_api_key = os.getenv("GOOGLE_API_KEY")
temperature = 0

# Define file and directory paths for input and output.
input_filename = "../data/leetcode_problems_processed_data.csv"
output_directory = f"output2/temperature-{temperature}/{llm_model}/"
results_directory = f"results2/temperature-{temperature}/"
time_limit = 60 # Timeout limit for executing generated scripts

# Ensure the output and results directories exist.
Path(output_directory).mkdir(parents=True, exist_ok=True)
Path(results_directory).mkdir(parents=True, exist_ok=True)

# Initialize the Google chat model
chat = ChatGoogleGenerativeAI(temperature=temperature, google_api_key=google_api_key, model=llm_model)

Step 3: Configure Model Settings for OpenAI via LangChain

In [ ]:
# List of available OpenAI models; select a specific model for processing.
llm_models = ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o", "gpt-4-turbo"]
llm_model = llm_models[3]
temperature = 1
time_limit = 60 # Timeout limit for executing generated scripts

# Update file and directory paths for the OpenAI model configuration.
input_filename = r"test_dataset/data/processed/leetcode_problems_processed_data.csv"  # Input CSV file with problems
output_directory = rf"test_dataset/outputs/outputs/py_files_outputs_v2/temperature-{temperature}/{llm_model}/"  # Directory for Python file outputs
results_directory = rf"test_dataset/outputs/visualizations/csv/results2/temperature-{temperature}/"  # Directory for CSV results

# Ensure the output and results directories exist.
Path(output_directory).mkdir(parents=True, exist_ok=True)
Path(results_directory).mkdir(parents=True, exist_ok=True)

# Set up the OpenAI API key and initialize the chat model via LangChain.
openai_api_key = os.getenv("OPENAI_API_KEY")
chat = ChatOpenAI(api_key=openai_api_key, model_name=llm_model, temperature=temperature)


Step 4: Define Response Schemas and Prompt Template

In [ ]:
# Define structured response schemas to parse the model's output.
response_schemas = [
    ResponseSchema(
        name="problem_solution",
        description="Code a solution in Python for the given problem. "
            "The solution should handle example inputs and verify if each output matches the expected result. "
            "Print for each input the value `True` if the output is correct for that input, otherwise `False`. "
            "Retun the solution as a Python program. "
    ),
    ResponseSchema(
        name="input_1",
        description="Extract the first example input from the problem description. "
                    "Output it formatted as input in a Python program. "
                    "Do not answer if this information is not found."
    ),
    ResponseSchema(
        name="output_1",
        description="Extract the first example output for comparison in a Python program. "
                    "Do not answer if this information is not found."
    ),
    ResponseSchema(
        name="input_2",
        description="Extract the second example input from the problem description. "
                    "Output it formatted as input in a Python program. "
                    "Do not answer if this information is not found."
    ),
    ResponseSchema(
        name="output_2",
        description="Extract the second example output for comparison in a Python program. "
                    "Do not answer if this information is not found."
    ),
    ResponseSchema(
        name="input_3",
        description="Extract the third example input from the problem description. "
                    "Output it formatted as input in a Python program. "
                    "Do not answer if this information is not found."
    ),
    ResponseSchema(
        name="output_3",
        description="Extract the third example output for comparison in a Python program. "
                    "Do not answer if this information is not found."
    ),
]

# Create a structured output parser using the defined schemas.
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

# Define the prompt template for interacting with the LLM.
solution_template = """
For the following problem description, extract the following information and format it as JSON:

- "Input 1"
- "Output 1"
- "Input 2"
- "Output 2"
- "Input 3"
- "Output 3"
- "Problem Solution"

Text: {text}

{format_instructions}
"""

# Initialize the chat prompt template.
prompt = ChatPromptTemplate.from_template(template=solution_template)

Step 5: Extract Problems from Input CSV File

In [ ]:
# Read the processed problems from the input CSV file.
problems = []
with open(input_filename, mode='r', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Skip header line if present
    for fields in csv_reader:
        # Each problem is stored as a dictionary with an ID and a description.
        problem = {
            "ID": fields[0],
            "Description": fields[1],
        }
        problems.append(problem)

Step 6: Process and Solve Problems Using the LLM

In [ ]:
datos = []  # List to store the results for each problem
print("Initial Data: {}".format(datos))

In [ ]:
# Define the starting lap and the maximum number of problems to solve
start_lap = 0  # Starting index for processing problems start_lap + 1 
max_problems = 25  # Maximum number of problems to process

In [ ]:
# Process a limited set of problems from the CSV.
for lap in range(start_lap, min(len(problems), start_lap + max_problems)):
    print(f"Processing Problem {lap + 1} of {max_problems}")
    python_code = ""  # Initialize the code variable
    
    try:
        problem_text = problems[lap]["Description"]
        # Format the messages for the chat prompt using the problem description.
        messages = prompt.format_messages(text=problem_text, format_instructions=format_instructions)
        # Query the LLM using the formatted prompt.
        response = chat(messages)

        time_start = time.time()  # Record start time for processing

        # Parse the model response into a structured dictionary.
        output_dict = output_parser.parse(response.content)
        
        
        # Extract the Python code solution from the response.
        python_code = output_dict.get("problem_solution")
        # Process the code by removing potential header/footer markers.
        if "python" in python_code:
            code_lines = python_code.split('\n')[1:-1]
            python_code = '\n'.join(code_lines)
            

        # Save the processed Python code to a file.
        output_file = f"{output_directory}output_{lap + 1}.py"
        with open(output_file, mode='w', newline='', encoding='utf-8') as pyfile:
            pyfile.write(python_code)
        print(f"Python code saved successfully in {output_file}.")
        
        # Execute the saved Python script and capture its output.
        script_path = output_file

        try:
            proceso = subprocess.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            salida, _ = proceso.communicate(timeout=time_limit)
            resultado = salida.decode().strip()
            # Count occurrences of "True" and "False" in the execution output.
            ocurrencias_true = resultado.count("True")
            ocurrencias_false = resultado.count("False")
            
            datos.append({
                "ID": lap + 1,
                "code": python_code,
                "result": resultado,
                "true_count": ocurrencias_true,
                "false_count": ocurrencias_false
            })
        except subprocess.TimeoutExpired:
            proceso.kill()
            datos.append({
                "ID": lap + 1,
                "code": python_code,
                "result": "TIMEOUT",
                "true_count": "TIMEOUT",
                "false_count": "TIMEOUT"
            })
        except (SyntaxError, IndentationError, TypeError, ValueError, ImportError,
                AttributeError, KeyError, IndexError) as e:
            datos.append({
                "ID": lap + 1,
                "code": python_code,
                "result": str(e),
                "true_count": "ERROR",
                "false_count": "ERROR"
            })
        
        time_end = time.time()  # Record end time for processing
        time_elapsed = time_end - time_start
        
    except Exception as e:
        datos.append({
            "ID": lap + 1,
            "code": python_code,
            "result": str(e),
            "true_count": "ERROR",
            "false_count": "ERROR"
        })
    
    # Pause execution for a total of 15 seconds per iteration.
    if lap < start_lap + max_problems - 1:
        try:
            if time_elapsed < 15:
                time.sleep(15 - time_elapsed)
        except NameError:
            time.sleep(15)

In [ ]:
# Output final responses for debugging purposes.
print("Response: {}".format(response))
print("Messages: {}".format(messages))
print("Output Dictionary: {}".format(output_dict))

Step 7: Write the Results to a CSV File

In [ ]:
nombre_archivo = f"{results_directory}results_{llm_model}.csv"
encabezados = ["ID", "code", "result", "true_count", "false_count"]

# Ensure that the results directory exists.
Path(results_directory).mkdir(parents=True, exist_ok=True)

with open(nombre_archivo, mode='a', newline='', encoding='utf-8') as archivo_csv:
    escritor_csv = csv.DictWriter(archivo_csv, fieldnames=encabezados)
    if archivo_csv.tell() == 0:
        escritor_csv.writeheader()
    escritor_csv.writerows(datos)

print(f"CSV file '{nombre_archivo}' created successfully.")